In [53]:
import pandas
import numpy
import pygsheets
# import scipy.stats
# import datetime

import matplotlib
import matplotlib.pyplot as mp
import squarify

In [54]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1fNf3eK_nqgzXX6ijGlg9A4MPXAF4fh9ZmH3XnsrvS9A') # copy for 2022 report
#spreadsheet = gc.open_by_key('1tcS6Wd-Wp-LTDpLzFgJY_RSNDnbyubW3J_9HKIAys4A')
#spreadsheet = gc.open_by_key('1il2duWGhw5WTnlHBqzdM7b3VXCufecO9e4kGPkgy9f8') # copy for 2023 report
spreadsheet = gc.open_by_key('1hOuMDpznpTKQRJ9SQGWY1ktMR_qw0vVRARrAtgeUsYQ')

#spreadsheet[1] "Gas Pipelines" tab is the second index
terms_df_orig = spreadsheet.worksheet('title', 'Terminals').get_as_df(start='A3')
region_df_orig = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

In [55]:
region_df_orig_cleaned = region_df_orig.loc[(region_df_orig.Region!='--')&
                                            (region_df_orig.SubRegion!='--')]
multiindex_region_subregion = region_df_orig_cleaned.groupby(['Region','SubRegion'])['Country'].count().index
multiindex_region_subregion_country = region_df_orig_cleaned.groupby(['Region','SubRegion','Country'])['Country'].count().index

In [58]:
# replace all -- with nans
terms_df_orig.replace('--', numpy.nan, inplace=True)
# remove oil export terminals
terms_df_orig = terms_df_orig.loc[terms_df_orig['Fuel']=='LNG']
# remove anything without a wiki page
terms_df_orig = terms_df_orig.loc[terms_df_orig['Wiki']!='']
# remove N/A statuses
terms_df_orig = terms_df_orig.loc[terms_df_orig['Status']!='']

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_14911/97535879.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  terms_df_orig.replace('--', numpy.nan, inplace=True)


In [66]:
terms_df_touse = terms_df_orig.copy()

In [67]:
status_list = ['Proposed', 
               'Construction', 
               'Shelved', 
               'Cancelled', 
               'Operating', 
               'Idle', 
               'Mothballed', 
               'Retired']
country_list = sorted(set(region_df_orig['Country'].tolist()))
region_list = sorted(set(region_df_orig['Region'].tolist()))
if '--' in region_list:
    region_list.remove('--')
subregion_list = sorted(set(region_df_orig['SubRegion'].tolist()))
if '--' in subregion_list:
    subregion_list.remove('--')

In [68]:
terms_df_orig.shape

(1258, 82)

## table of all projects under construction or proposed

In [101]:
table_for_map

,TerminalID,TerminalName,UnitName,Status,Latitude,Longitude,CapacityInMtpa,StartYearEarliest,CountryISO3166-1alpha-3
Country,,,,,,,,,
Germany,T1077,Lubmin FSRU,Phase 2 (Vessel 1),Proposed,54.153332,13.645319,1.47,2023.0,DEU
Germany,T1077,Lubmin FSRU,Phase 2 (Vessel 2),Proposed,54.153332,13.645319,5.15,2024.0,DEU
Brazil,T0934,Presidente Kennedy FSRU,NaN,Proposed,-21.224754,-40.946933,5.37,NaN,BRA
Israel,T0829,NewMed FLNG Terminal,NaN,Proposed,32.614,34.808,5.15,NaN,ISR
Croatia,T0430,Krk FSRU,Phase 1,Proposed,45.188543,14.545081,2.35,2024.0,HRV
...,...,...,...,...,...,...,...,...,...
India,T1145,Gopalpur FSRU,Phase 2,Proposed,19.301786,84.966246,5.00,NaN,IND
India,T1146,AG&P India FSRU,NaN,Proposed,Unknown,Unknown,5.00,2024.0,IND
Indonesia,T1147,Madura FLNG Terminal,NaN,Proposed,-7.371,113.2767,NaN,NaN,IDN


In [104]:
table_for_map = terms_df_touse.loc[terms_df_touse.Status.isin(['Construction','Proposed'])][['TerminalID','TerminalName','UnitName','FacilityType','Status','Country','Latitude','Longitude','CapacityInMtpa','StartYearEarliest']]
table_for_map.replace('',numpy.nan,inplace=True)
table_for_map = table_for_map.set_index('Country')
table_for_map['CountryISO3166-1alpha-3'] = region_df_orig.set_index('Country').loc[table_for_map.index,'CountryISO3166-1alpha-3']
table_for_map.to_excel('table_for_map.xlsx', na_rep='NaN')

## big totals for top

In [106]:
terms_df_touse.loc[terms_df_touse.Status=='Operating'].CapacityInMtpa.sum()

np.float64(1532.92)

In [108]:
terms_df_touse.loc[terms_df_touse.Status.isin(['Proposed','Construction'])].CapacityInMtpa.sum()

np.float64(1622.29)

In [116]:
terms_df_touse.loc[(terms_df_touse.Status=='Operating')&
                   (terms_df_touse.StartYearEarliest==2023)].CapacityInMtpa.sum()

np.float64(82.80000000000001)

In [118]:
# shelved or cancelled in past 5 years (2019–2023)
terms_df_touse.loc[(terms_df_touse.Status.isin(['Shelved','Cancelled'])) &
                            (
                                terms_df_touse.ShelvedYear.isin(list(range(2019,2024))) |
                                terms_df_touse.CancelledYear.isin(list(range(2019,2024)))
                            )].CapacityInMtpa.sum()

np.float64(594.28)

## exports region/subregion

In [86]:
#mtpa by country

mtpa_by_country_df = pandas.DataFrame(columns=status_list, index=country_list)
mtpa_by_region_df = pandas.DataFrame(columns=status_list, index=multiindex_region_subregion)

for status in status_list:
    #print(status)
    temp_df = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                                   (terms_df_orig['FacilityType']=='Export')&
                                                   (terms_df_orig.Status==status)]
    mtpa_by_country_df[status] = temp_df.groupby('Country')['CapacityInMtpa'].sum()

for status in status_list:
    #print(status)
    temp_df = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                                  (terms_df_orig['FacilityType']=='Export')&
                                                  (terms_df_orig.Status==status)]
    mtpa_by_region_df[status] = temp_df.groupby(['Region','SubRegion'])['CapacityInMtpa'].sum()

mtpa_by_country_df = mtpa_by_country_df.fillna(0)
mtpa_by_region_df = mtpa_by_region_df.fillna(0)

# add total and proposed+construction info
mtpa_by_region_df['Proposed+Construction'] = mtpa_by_region_df[['Proposed','Construction']].sum(axis=1)
mtpa_by_region_df.sort_values(by='Proposed+Construction', inplace=True)
mtpa_by_region_df = mtpa_by_region_df[['Proposed', 'Construction', 'Proposed+Construction', 'Shelved', 'Cancelled', 'Operating', 'Idle', 'Mothballed', 'Retired']]
mtpa_by_region_df.sort_values('Proposed+Construction', ascending=False, inplace=True)

mtpa_by_country_df['Proposed+Construction'] = mtpa_by_country_df[['Proposed','Construction']].sum(axis=1)
mtpa_by_country_df.sort_values(by='Proposed+Construction', inplace=True)
mtpa_by_country_df = mtpa_by_country_df[['Proposed', 'Construction', 'Proposed+Construction', 'Shelved', 'Cancelled', 'Operating', 'Idle', 'Mothballed', 'Retired']]
mtpa_by_country_df.sort_values('Proposed+Construction', ascending=False, inplace=True)

mtpa_by_region_df.index.names = ['Region','Subregion']
mtpa_by_region_df.sort_index(level='Region', inplace=True)

mtpa_by_region_df.to_excel('global-export-capacity-totals.xlsx')
mtpa_by_region_df = mtpa_by_region_df.loc[~(mtpa_by_region_df==0).all(axis=1)]

# total
mtpa_by_region_df.loc['Total',:] = mtpa_by_region_df.sum(axis=0).values
mtpa_by_country_df.loc['Total',:] = mtpa_by_country_df.sum(axis=0).values

# save countries
mtpa_by_country_df = mtpa_by_country_df.loc[~(mtpa_by_country_df==0).all(axis=1)]
mtpa_by_country_df.to_excel('mtpa-by-country-export.xlsx')

# mtpa_by_region_df = mtpa_by_region_df.loc[~(mtpa_by_region_df==0).all(axis=1)]
mtpa_by_region_export_df = mtpa_by_region_df.copy()

mtpa_by_region_df

Proposed  Construction  \
Region   Subregion                                                 
Africa   Northern Africa                      0.00          0.07   
         Sub-Saharan Africa                 102.55         13.52   
Americas Latin America and the Caribbean     89.56          7.45   
         Northern America                   322.58         90.11   
Asia     Central Asia                         0.00          0.00   
         South-eastern Asia                  16.70          0.00   
         Southern Asia                        0.00         10.80   
         Western Asia                        30.75         33.00   
Europe   Eastern Europe                     131.40         32.70   
         Northern Europe                      0.00          0.00   
Oceania  Australia and New Zealand           21.70          5.00   
         Melanesia                            9.20          0.00   
Total                                       724.44        192.65   

                                          Proposed+Construction  Shelved  \
Region   Subregion                                                         
Africa   Northern Africa                                   0.07      0.0   
         Sub-Saharan Africa                              116.07     10.0   
Americas Latin America and the Caribbean                  97.01      0.0   
         Northern America                                412.69      6.0   
Asia     Central Asia                                      0.00      0.0   
         South-eastern Asia                               16.70      0.0   
         Southern Asia                                    10.80      0.0   
         Western Asia                                     63.75     22.0   
Europe   Eastern Europe                                  164.10     16.2   
         Northern Europe                                   0.00      0.0   
Oceania  Australia and New Zealand                        26.70      7.2   
         Melanesia                                         9.20      1.5   
Total                                                    917.09     62.9   

                                          Cancelled  Operating   Idle  \
Region   Subregion                                                      
Africa   Northern Africa                       5.00      37.54   0.00   
         Sub-Saharan Africa                   27.39      37.72   0.00   
Americas Latin America and the Caribbean      21.60      16.45   0.00   
         Northern America                    460.82      92.92   0.00   
Asia     Central Asia                          0.00       0.20   0.00   
         South-eastern Asia                    2.95      52.55   9.60   
         Southern Asia                        63.95       0.00   0.00   
         Western Asia                         22.64      95.40   0.00   
Europe   Eastern Europe                       16.36      31.06   3.76   
         Northern Europe                       0.00       4.67   0.00   
Oceania  Australia and New Zealand            45.80      87.60   0.00   
         Melanesia                             6.00       8.30   0.00   
Total                                        672.51     464.41  13.36   

                                          Mothballed  Retired  
Region   Subregion                                             
Africa   Northern Africa                        3.20     7.80  
         Sub-Saharan Africa                     0.00     0.00  
Americas Latin America and the Caribbean        3.75     0.00  
         Northern America                       1.50     0.00  
Asia     Central Asia                           0.00     0.00  
         South-eastern Asia                     0.00    30.10  
         Southern Asia                          0.00     0.00  
         Western Asia                           7.20     0.00  
Europe   Eastern Europe                         0.00     0.00  
         Northern Europe                        0.00     0.01  
Oceania  Austral

In [87]:
mtpa_by_country_df.to_excel('mtpa_by_country_imports.xlsx')

## imports region/subregion

In [82]:
#mtpa by country

mtpa_by_country_df = pandas.DataFrame(columns=status_list, index=country_list)
mtpa_by_region_df = pandas.DataFrame(columns=status_list, index=multiindex_region_subregion)

for status in status_list:
    #print(status)
    temp_df = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                                   (terms_df_orig['FacilityType']=='Import')&
                                                   (terms_df_orig.Status==status)]
    mtpa_by_country_df[status] = temp_df.groupby('Country')['CapacityInMtpa'].sum()

for status in status_list:
    #print(status)
    temp_df = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                                  (terms_df_orig['FacilityType']=='Import')&
                                                  (terms_df_orig.Status==status)]
    mtpa_by_region_df[status] = temp_df.groupby(['Region','SubRegion'])['CapacityInMtpa'].sum()

mtpa_by_country_df = mtpa_by_country_df.fillna(0)
mtpa_by_region_df = mtpa_by_region_df.fillna(0)

# add total and proposed+construction info
mtpa_by_region_df['Proposed+Construction'] = mtpa_by_region_df[['Proposed','Construction']].sum(axis=1)
mtpa_by_region_df.sort_values(by='Proposed+Construction', inplace=True)
mtpa_by_region_df = mtpa_by_region_df[['Proposed', 'Construction', 'Proposed+Construction', 'Shelved', 'Cancelled', 'Operating', 'Idle', 'Mothballed', 'Retired']]
mtpa_by_region_df.sort_values('Proposed+Construction', ascending=False, inplace=True)

mtpa_by_country_df['Proposed+Construction'] = mtpa_by_country_df[['Proposed','Construction']].sum(axis=1)
mtpa_by_country_df.sort_values(by='Proposed+Construction', inplace=True)
mtpa_by_country_df = mtpa_by_country_df[['Proposed', 'Construction', 'Proposed+Construction', 'Shelved', 'Cancelled', 'Operating', 'Idle', 'Mothballed', 'Retired']]
mtpa_by_country_df.sort_values('Proposed+Construction', ascending=False, inplace=True)

mtpa_by_region_df.index.names = ['Region','Subregion']
mtpa_by_region_df.sort_index(level='Region', inplace=True)

mtpa_by_region_df.to_excel('global-import-capacity-totals.xlsx')
mtpa_by_region_df = mtpa_by_region_df.loc[~(mtpa_by_region_df==0).all(axis=1)]

# total
mtpa_by_region_df.loc['Total',:] = mtpa_by_region_df.sum(axis=0).values
mtpa_by_country_df.loc['Total',:] = mtpa_by_country_df.sum(axis=0).values

# save countries
mtpa_by_country_df = mtpa_by_country_df.loc[~(mtpa_by_country_df==0).all(axis=1)]
mtpa_by_country_df.to_excel('mtpa-by-country-export.xlsx')

# mtpa_by_region_df = mtpa_by_region_df.loc[~(mtpa_by_region_df==0).all(axis=1)]
mtpa_by_region_import_df = mtpa_by_region_df.copy()

mtpa_by_region_df

Proposed  Construction  \
Region   Subregion                                                 
Africa   Northern Africa                      0.00          0.00   
         Sub-Saharan Africa                   1.50          1.70   
Americas Latin America and the Caribbean     40.88         16.06   
         Northern America                     0.05          0.00   
Asia     Eastern Asia                       195.15        113.20   
         South-eastern Asia                  38.00          8.60   
         Southern Asia                       67.36         30.00   
         Western Asia                         0.00          2.00   
Europe   Eastern Europe                       4.49          1.54   
         Northern Europe                     34.37          3.80   
         Southern Europe                     45.95          4.04   
         Western Europe                      66.64         19.24   
Oceania  Australia and New Zealand            8.30          2.33   
Total                                       502.69        202.51   

                                          Proposed+Construction  Shelved  \
Region   Subregion                                                         
Africa   Northern Africa                                   0.00     5.15   
         Sub-Saharan Africa                                3.20     0.88   
Americas Latin America and the Caribbean                  56.94    16.28   
         Northern America                                  0.05     0.00   
Asia     Eastern Asia                                    308.35    40.26   
         South-eastern Asia                               46.60    27.90   
         Southern Asia                                    97.36    25.54   
         Western Asia                                      2.00    11.10   
Europe   Eastern Europe                                    6.03     9.31   
         Northern Europe                                  38.17     0.00   
         Southern Europe                                  49.99     3.65   
         Western Europe                                   85.88     0.00   
Oceania  Australia and New Zealand                        10.63     1.00   
Total                                                    705.20   141.07   

                                          Cancelled  Operating  Idle  \
Region   Subregion                                                     
Africa   Northern Africa                      10.64       0.00   5.7   
         Sub-Saharan Africa                    7.70       2.50   0.0   
Americas Latin America and the Caribbean      24.97      77.14   0.0   
         Northern America                    259.52      59.80  25.0   
Asia     Eastern Asia                         52.17     532.35   0.0   
         South-eastern Asia                   35.17      58.15   0.0   
         Southern Asia                        72.45      65.50   0.0   
         Western Asia                          8.00      72.80  22.9   
Europe   Eastern Europe                        7.30       4.58   0.0   
         Northern Europe                      27.16      43.78   0.0   
         Southern Europe                      47.68      79.91   0.0   
         Western Europe                       12.40      72.00   0.0   
Oceania  Australia and New Zealand             1.75       0.00   0.0   
Total                                        566.91    1068.51  53.6   

                                          Mothballed  Retired  
Region   Subregion                                             
Africa   Northern Africa                         4.2      0.0  
         Sub-Saharan Africa                      0.0      0.0  
Americas Latin America and the Caribbean         0.0      0.0  
         Northern America                       45.0     17.3  
Asia     Eastern Asia                            1.5      0.0  
         South-eastern Asia                      0.4      0.0  
         Southern Asia                           0.0      0.0  
       

In [83]:
mtpa_by_country_df.to_excel('mtpa_by_country_exports.xlsx')

## import/export capacity over time

### import

In [84]:
mtpa_started_sum = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                     (terms_df_orig['FacilityType']=='Import')&
                                     (terms_df_orig.Status=='Operating')].groupby('StartYearEarliest')['CapacityInMtpa'].sum()

mtpa_proposed_sum = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                     (terms_df_orig['FacilityType']=='Import')&
                                     (terms_df_orig.Status=='Proposed')].groupby('ProposalYear')['CapacityInMtpa'].sum()

mtpa_construction_sum = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                     (terms_df_orig['FacilityType']=='Import')&
                                     (terms_df_orig.Status=='Construction')].groupby('ConstructionYear')['CapacityInMtpa'].sum()

mtpa_by_start_year = pandas.DataFrame(index=list(range(1980,2025)))
mtpa_by_start_year.index.name = 'Start year'
mtpa_by_start_year['Import mtpa operating'] = mtpa_started_sum
mtpa_by_start_year['Import mtpa construction'] = mtpa_construction_sum
mtpa_by_start_year['Import mtpa proposed'] = mtpa_proposed_sum
mtpa_by_start_year.replace(numpy.nan,0,inplace=True)

mtpa_by_start_year.to_excel('mtpa-by-start-year-import.xlsx')

In [85]:
mtpa_started_sum = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                     (terms_df_orig['FacilityType']=='Export')&
                                     (terms_df_orig.Status=='Operating')].groupby('StartYearEarliest')['CapacityInMtpa'].sum()

mtpa_proposed_sum = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                     (terms_df_orig['FacilityType']=='Export')&
                                     (terms_df_orig.Status=='Proposed')].groupby('ProposalYear')['CapacityInMtpa'].sum()

mtpa_construction_sum = terms_df_orig.loc[(terms_df_orig.Fuel=='LNG')&
                                     (terms_df_orig['FacilityType']=='Export')&
                                     (terms_df_orig.Status=='Construction')].groupby('ConstructionYear')['CapacityInMtpa'].sum()

mtpa_by_start_year = pandas.DataFrame(index=list(range(1980,2025)))
mtpa_by_start_year.index.name = 'Start year'
mtpa_by_start_year['Export mtpa operating'] = mtpa_started_sum
mtpa_by_start_year['Export mtpa construction'] = mtpa_construction_sum
mtpa_by_start_year['Export mtpa proposed'] = mtpa_proposed_sum
mtpa_by_start_year.replace(numpy.nan,0,inplace=True)

mtpa_by_start_year.to_excel('mtpa-by-start-year-export.xlsx')

# capacity by country

## export

In [ ]:
terms_df_subset = terms_df_touse.copy()[(terms_df_touse['FacilityType']=='Import') & (terms_df_touse['Fuel']=='LNG')]

cap_by_country = pandas.DataFrame(0, columns=status_list, index=country_list)
cap_by_region = pandas.DataFrame(0, columns=status_list, index=multiindex_region_subregion)

print('===country-level calculations===')
for status in status_list:
    print(status)
    terms_df_subset_status = terms_df_subset.copy()[terms_df_subset['Status']==status]
    cap_by_country[status] = terms_df_subset_status.groupby('Country')['CapacityInMtpa'].sum()

print('===country-level calculations===')
for status in status_list:
    print(status)
    terms_df_subset_status = terms_df_subset.copy()[terms_df_subset['Status']==status]
    cap_by_region[status] = terms_df_subset_status.groupby(['Region','SubRegion'])['CapacityInMtpa'].sum()

#fille NaN with 0.0
cap_by_region = cap_by_region.fillna(0)
cap_by_country = cap_by_country.fillna(0)

cap_by_region['In Development (Proposed + Construction)'] = cap_by_region[['Proposed','Construction']].sum(axis=1)
cap_by_country['In Development (Proposed + Construction)'] = cap_by_country[['Proposed','Construction']].sum(axis=1)

cap_by_country = cap_by_country[excel_status_list]
cap_by_region = cap_by_region[excel_status_list]

cap_by_region.index.names = ['Region','Subregion']
cap_by_country.index.name = 'Country'

cap_by_region.loc['Total',:] = cap_by_region.sum(axis=0).values
cap_by_country.loc['Total',:] = cap_by_country.sum(axis=0).values

#cap_by_region = cap_by_region.loc[~(cap_by_region==0).all(axis=1)]
cap_by_country = cap_by_country.loc[~(cap_by_country==0).all(axis=1)]

cap_by_region.replace(0,'',inplace=True)
cap_by_country.replace(0,'',inplace=True)

cap_by_region.to_excel(excel_writer, 'LNG import capacity by region')
cap_by_country.to_excel(excel_writer, 'LNG import capacity by country')